In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
from extract import endpoints, get
from transform import (transform_sports, transform_leagues, transform_seasons,
                       transform_divisions, transform_teams,TeamsBySeasonSchema)
from transform import (DivisionSchema, DivisionSeasonsSchema,SportSchema, LeagueSchema,
                       SeasonSchema, TeamSchema)
from transform import SportsSeasons, LeagueCollection
from pathlib import Path

In [ ]:
data = Path('data/statsapi')
tables = endpoints.keys()
paths = [data/(table+'.parquet') for table in tables]

In [ ]:
for table, path in zip(tables, paths):
    if not path.exists():
        get(table).collect().write_parquet(path)

In [ ]:
sports = pl.scan_parquet(data/'sports.parquet')
leagues = pl.scan_parquet(data/'leagues.parquet')
divisions = pl.scan_parquet(data/'divisions.parquet')
seasons = pl.scan_parquet(data/'seasons.parquet')
teams = pl.scan_parquet(data/'teams.parquet')

In [ ]:
sports = transform_sports(sports)
sports = SportSchema.validate(sports, cast=True).lazy()

In [ ]:
leagues = transform_leagues(leagues)
leagues = LeagueSchema.validate(leagues, cast=True).lazy()

In [ ]:
seasons  = transform_seasons(seasons)
seasons = SeasonSchema.validate(seasons, cast=True).lazy()

In [ ]:
SC, bad = SportsSeasons.filter(
    {
        'sports': sports,
        'seasons': seasons,
    }
)

In [ ]:
bad['sports']._df

In [ ]:
div, div_seasons = transform_divisions(divisions, leagues)
divisions = DivisionSchema.validate(div, cast=True).lazy()
division_seasons = DivisionSeasonsSchema.validate(div_seasons, cast=True).lazy()

In [ ]:
teams, team_seasons = transform_teams(teams)
teams = TeamSchema.validate(teams, cast=True).lazy()
team_seasons = TeamsBySeasonSchema.validate(team_seasons, cast=True).lazy()

In [ ]:
LC = LeagueCollection.validate(
    {
        'leagues': leagues,
        'divisions': divisions,
        'seasons': SC.seasons,
        'team_seasons': team_seasons
    }
)